# Diagrama explicativo del protocolo SPI
- Clock -> director de la orquesta para saber cuando hablar
- CS -> Chip select -> Para pellizcar al que le voy a hablar
- MISO -> Master Input - Slave Output
- MOSI -> Master output - Slave Input

- 00001010 (LSB -> Least significar byte)
- 0101000  (MSB -> Most significant byte)

![Diagrama de clase del protocolo SPI](https://i.imgur.com/0UlN7vz.png "Diagrama de clase del protocolo SPI")

# Circuito de la termocupla

El siguiente circuito va de la mano con el siguiente código que desarrollamos en clase para el uso del mismo. En este caso se trabajo con las llantas por decirlo de una manera, ya que en el mundo profesional estas librerias ya existen así que el código que desarrollamos manualmente viene por parte de un driver que se utiliza para usar estos sensores; se hizo de est manera para buscar la compresión de cómo funciona por detrás y además de entender el *protocolo SPI*.

![Circuito con la termocupla](https://i.imgur.com/KnlkGko.jpeg "Circuito con la termocupla")

In [ ]:
from machine import SoftSPI, Pin
import time

class Thermocouple:
    def __init__(self, miso_pin = 12, sck_pin = 14, mosi_pin = 13, chip_select_pin = 15):
        self.miso_pin = miso_pin
        self.sck_pin = sck_pin
        self.mosi_pin = mosi_pin
        self.chip_select_pin = chip_select_pin
        self.bus = SoftSPI(sck = Pin(self.sck_pin), mosi = Pin(self.mosi_pin), miso = Pin(self.miso_pin))
        
        self.chip_select = Pin(self.chip_select_pin, Pin.OUT)
        self.chip_select.value(1)
        
    def measure(self):
        self.chip_select.value(1)
        time.sleep(1)
        self.chip_select.value(0)
        time.sleep(1e-6) # Esperar mil nanosegundos?
        response = self.bus.read(2)
        time.sleep(1e-6)
        self.chip_select.value(1)
        
        response = list(response)
        print(f"[INFO] Response: {response}")
        
        byte_high = response[0]<<8 #'<<' Correr 8 posiciones a la derecha (agrega 8 0's a la derecha del numero)
        byte_low = response[1]
        bin_response = byte_high + byte_low
        bin_response = bin_response>>3 #Agrega ceros a la izquierda lo que corre el numero a la derecha
        
        temperature = bin_response/4
        print(f"[INFO] Temperature: {temperature}")
        
        
        
if __name__ == "__main__":
    sensor = Thermocouple()
    while True:
        sensor.measure()